# Receipt OCR Preprocessing Pipeline
Visualise each step of the preprocessing pipeline from `OCRExtractor._preprocess()`.
Compare raw Tesseract vs. preprocessed Tesseract output.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "python-worker"))

import cv2
import numpy as np
import pytesseract
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.bbox"] = "tight"

print("OpenCV:", cv2.__version__)
print("Tesseract:", pytesseract.get_tesseract_version())

## 1. Pick an image to analyse

In [ ]:
# List available images in storage/uploads
upload_dir = "storage/uploads"
images = [f for f in os.listdir(upload_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print("Available images:")
for i, img in enumerate(images):
    print(f"  [{i}] {img}")

# Change this to pick a different image
IMAGE_INDEX = 0
IMAGE_PATH = os.path.join(upload_dir, images[IMAGE_INDEX])
print(f"\nUsing: {IMAGE_PATH}")

## 2. Raw Tesseract (no preprocessing) — baseline

In [ ]:
raw_image = Image.open(IMAGE_PATH)
print("Image size:", raw_image.size, "mode:", raw_image.mode)

# Default Tesseract (PSM 3, no config)
raw_text = pytesseract.image_to_string(raw_image)
print("--- Raw Tesseract output ---")
print(raw_text[:800] if len(raw_text) > 800 else raw_text)

## 3. Step-by-step preprocessing

In [ ]:
def show_image(img, title, cmap=None, bgr=False):
    """Display an image with consistent sizing."""
    if bgr:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(8, 8 * h / w))
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=12)
    ax.axis("off")
    plt.show()

In [ ]:
# Load via OpenCV
orig = cv2.imread(IMAGE_PATH)
orig_h, orig_w = orig.shape[:2]
print(f"Original dimensions: {orig_w}x{orig_h}")
show_image(orig, "1. Original image", bgr=True)

In [ ]:
# Step 1: Resize to 500px width
TARGET_WIDTH = 500
h, w = orig.shape[:2]

if w > TARGET_WIDTH:
    ratio = TARGET_WIDTH / float(w)
    new_h = int(h * ratio)
    resized = cv2.resize(orig, (TARGET_WIDTH, new_h), interpolation=cv2.INTER_AREA)
else:
    resized = orig.copy()
    ratio = 1.0

ratio = orig.shape[1] / float(resized.shape[1])
print(f"Resize ratio: {ratio:.3f}")
print(f"Resized dimensions: {resized.shape[1]}x{resized.shape[0]}")
show_image(resized, "2. Resized to 500px width", bgr=True)

In [ ]:
# Step 2: Grayscale
gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
show_image(gray, "3. Grayscale", cmap="gray")

In [ ]:
# Step 3: Gaussian blur
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
show_image(blurred, "4. Gaussian blur (5x5)", cmap="gray")

In [ ]:
# Step 4: Canny edge detection
edged = cv2.Canny(blurred, 75, 200)
show_image(edged, "5. Canny edge detection (75, 200)", cmap="gray")

In [ ]:
# Step 5: Find contours — draw ALL significant contours to see what we're working with
cnts = cv2.findContours(edged.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts = cnts[0] if len(cnts) == 2 else cnts[1]
cnts = sorted(cnts, key=cv2.contourArea, reverse=True)

print(f"Found {len(cnts)} contours")
for i, c in enumerate(cnts[:5]):
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)
    print(f"  Contour {i}: area={area:.0f}, vertices={len(approx)}")

# Visualise contours on the resized image
contour_viz = cv2.cvtColor(resized.copy(), cv2.COLOR_BGR2RGB)
cv2.drawContours(contour_viz, cnts[:10], -1, (0, 255, 0), 2)
show_image(contour_viz, "6. Contours (top 10, green)")

In [ ]:
# Step 6: Find receipt contour (largest 4-point polygon)
receipt_contour = None
for c in cnts:
    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)
    if len(approx) == 4:
        receipt_contour = approx
        break

if receipt_contour is not None:
    # Draw the found receipt boundary
    receipt_viz = cv2.cvtColor(resized.copy(), cv2.COLOR_BGR2RGB)
    cv2.drawContours(receipt_viz, [receipt_contour], -1, (255, 0, 0), 3)
    show_image(receipt_viz, "6a. Receipt boundary found (blue)")
    print("Receipt contour found ✓")
else:
    print("No 4-point receipt contour found — will fall back to grayscale")
    show_image(resized, "6a. No receipt found — fallback")

In [ ]:
def four_point_transform(image, pts):
    """Apply perspective transform (same as OCRExtractor._four_point_transform)."""
    def order_points(pts):
        rect = np.zeros((4, 2), dtype="float32")
        s = pts.sum(axis=1)
        rect[0] = pts[np.argmin(s)]
        rect[2] = pts[np.argmax(s)]
        diff = np.diff(pts, axis=1)
        rect[1] = pts[np.argmin(diff)]
        rect[3] = pts[np.argmax(diff)]
        return rect

    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    width_a = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    width_b = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    max_width = max(int(width_a), int(width_b))

    height_a = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    height_b = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    max_height = max(int(height_a), int(height_b))

    dst = np.array(
        [[0, 0], [max_width - 1, 0], [max_width - 1, max_height - 1], [0, max_height - 1]],
        dtype="float32",
    )

    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(image, M, (max_width, max_height))


# Step 7: Apply perspective transform or fall back to grayscale
if receipt_contour is not None:
    scaled_pts = receipt_contour.reshape(4, 2) * ratio
    warped = four_point_transform(orig, scaled_pts)
    show_image(warped, "7. Perspective transform (deskewed)", bgr=True)
    print(f"Deskewed dimensions: {warped.shape[1]}x{warped.shape[0]}")
else:
    warped = cv2.cvtColor(orig, cv2.COLOR_BGR2GRAY)
    show_image(warped, "7. Fallback: grayscale", cmap="gray")

In [ ]:
# Step 8: Prepare for Tesseract (BGR -> RGB if 3-channel, else keep grayscale)
if len(warped.shape) == 3:
    preprocessed = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)
else:
    preprocessed = warped

show_image(preprocessed, "8. Final preprocessed image (ready for Tesseract)",
           cmap="gray" if len(preprocessed.shape) == 2 else None)

## 4. Compare: Preprocessed vs Raw Tesseract output

In [ ]:
# Preprocessed OCR with --psm 6
preprocessed_text = pytesseract.image_to_string(preprocessed, config="--psm 6")
print("=== PREPROCESSED + PSM 6 ===")
print(preprocessed_text[:800] if len(preprocessed_text) > 800 else preprocessed_text)
print()
print("=== RAW TESSERACT (no preprocessing, default PSM) ===")
print(raw_text[:800] if len(raw_text) > 800 else raw_text)

## 5. Test the actual OCRExtractor (end-to-end)

In [ ]:
from src.extractors.ocr_extractor import OCRExtractor

extractor = OCRExtractor()
result = extractor.extract(IMAGE_PATH)
print("=== OCRExtractor.extract() ===")
print(result[:800] if len(result) > 800 else result)

## 6. Tune Canny thresholds interactively

In [ ]:
from ipywidgets import interact, IntSlider

def try_canny(low=75, high=200):
    edged = cv2.Canny(blurred, low, high)
    show_image(edged, f"Canny({low}, {high})", cmap="gray")

interact(try_canny, low=IntSlider(10, 200, 75), high=IntSlider(50, 300, 200))